# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data: 
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [5]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Healthcare / MedTech," which appears three times among the listed projects.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided documents, there are no specific use cases explicitly focused on security. However, there is a mention of security in one project related to healthcare: "Pathfinder 24" with the description indicating an AI-powered platform optimizing logistics routes for sustainability, and a secondary domain of security. But the primary focus appears to be on logistics rather than security per se. \n\nOverall, no direct or detailed security use cases are highlighted in the provided context.'

In [13]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally had positive comments about the fintech projects. They described some as clever solutions with measurable environmental benefits, promising ideas with robust experimental validation, and solid work with impressive real-world impact. For example, one project was praised as a "clever solution with measurable environmental benefit," and others were noted for their strong technical execution and real-world applicability.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Finance / FinTech," which is mentioned multiple times.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. The project titled "SecureNest 49" involves a document summarization and retrieval system for enterprise knowledge bases, which is in the E‑commerce / Marketplaces domain with a secondary focus on Legal / Compliance.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects were generally positive. For the project "SynthMind," judges found the concept to be strong but noted that results need more benchmarking. Overall, the feedback indicates recognition of the projects\' conceptual strength, although some areas for improvement were identified.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

I can think of an example from healthcare domain where BM25 outperforms embeddings.
Let's say we have a database with these NDC codes and their descriptions:

12345-678-90 -> "Lisinopril 10mg Tablet"

12345-678-99 -> "Lisinopril 20mg Tablet"

How Semantic Search Would Work:

The embedding model would convert all NDC codes to vectors

12345-678-90 and 12345-678-99 would have very similar vector representations because they share most of their characters

The model might return BOTH Lisinopril 10mg AND Lisinopril 20mg as "similar results"

This is medically dangerous - 10mg vs 20mg is a critical difference!

How BM25 Excels:

BM25 treats NDC codes as exact strings

Only the document containing the exact string 12345-678-90 would match

Returns the precise description: "Lisinopril 10mg Tablet"

And in a similar way, 12345-678-90 and 99999-999-99 are "distant" as strings but might both represent cardiovascular drugs.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Security," as it is mentioned for at least one project in the sample. However, since only a few projects are shown and no comprehensive analysis of the entire dataset is provided, I cannot definitively determine the most common domain overall. If the dataset contains many projects across various domains, a full analysis would be needed to identify the most frequent one accurately.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided data, there are no specific use cases related to security explicitly mentioned. The use cases listed focus on privacy improvements in healthcare applications through federated learning, rather than directly addressing security.\n\nIf you need further assistance or clarification, please let me know!'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects. For example, they praised "Pathfinder 27" for its excellent code quality and use of open-source libraries, awarding it a judge score of 9.8. Additionally, "PlanPilot 35," which is related to privacy in healthcare applications within the fintech domain, received a judge score of 8.4 and was recognized for being a clever solution with measurable environmental benefits.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [25]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "E-commerce / Marketplaces."'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security mentioned in the context. Specifically, one project titled "SecureNest" focuses on a document summarization and retrieval system for enterprise knowledge bases, which is related to legal and compliance security aspects. Additionally, other projects like "LearnWise" involve AI model compression for on-device reasoning, which can have implications for security in IoT sensors, though this is less explicitly stated.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had various positive comments about the fintech projects. For example, one judge described the project as "A clever solution with measurable environmental benefit," indicating appreciation for its innovative approach and potential impact. Another found it "Technically ambitious and well-executed," highlighting the project\'s strong technical foundation. Additionally, a project was praised as "Promising idea with robust experimental validation," showing confidence in its potential. Overall, the judges recognized the fintech projects for their innovation, technical quality, and potential real-world impact.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

The primary goal of using the multi query retriever is to capture diffferent perspectives of the same query, so we will have more chances of covering a wider net not to miss any retrieving documents that would be missed by a single query.

Example Domain: Corporate Sustainability Reporting
Original User Query:
"How are companies reducing their carbon footprint in manufacturing?"

Generated Reformulations (via LLM):
"Strategies for decreasing greenhouse gas emissions in industrial production"

"Manufacturing process modifications to lower CO2 output"

"Corporate decarbonization approaches in factory operations"

"Ways industrial companies are minimizing their environmental impact"

"Best practices for carbon emission reduction in manufacturing facilities"

Why This Improves Recall
1. Vocabulary Mismatch Resolution
Original query might only find documents using "carbon footprint"

Reformulation #1 finds documents using "greenhouse gas emissions" instead

Reformulation #3 catches documents using the newer term "decarbonization"

2. Conceptual Expansion
Original query focuses narrowly on "reducing"

Reformulation #4 broadens to "minimizing environmental impact," catching documents that discuss carbon reduction as part of broader sustainability initiatives

Reformulation #5 introduces "best practices," capturing guideline and framework documents

3. Technical vs. Business Language
Original query uses common business terminology

Reformulation #2 uses more technical language ("process modifications," "CO2 output") that might appear in engineering reports or technical white papers

4. Scope Variation
Original query specifies "manufacturing"

Reformulation #1 uses "industrial production," which might include mining, energy production, or other industrial activities with relevant transferable strategies

Multi-query retrieval is particularly valuable when:

Dealing with technical domains with specialized vocabulary

Users may not know the precise terminology used in the corpus

The document collection uses diverse language and phrasing

High recall is more important than perfect precision

##### ✅ Answer


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned in one of the project\'s primary domains. However, since the data includes only a few sample entries, I cannot definitively determine if it is the most common overall. If these entries are representative, then "Healthcare / MedTech" would be the most common project domain.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit usecases specifically about security. However, there are several projects involving federated learning aimed at improving privacy in healthcare applications, which relates to data security and privacy.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive remarks about the fintech projects. For example, one judge described the solution as "a clever solution with measurable environmental benefit," indicating appreciation for innovation and impact. Another project received comments highlighting it as a "comprehensive and technically mature approach," while another was considered promising with "robust experimental validation." Overall, the judges recognized the projects as technically ambitious, well-executed, and promising.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain based on the provided data appears to be "E‑commerce / Marketplaces," which is mentioned multiple times in the list of projects.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the project titled "LearnWise" involves an AI model compression suite enabling on-device reasoning for IoT sensors, which is relevant to security in IoT devices by enhancing privacy and local data processing.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had a generally positive view of the fintech projects. For example, the project "SynthMind," an AI-powered platform optimizing logistics routes for sustainability, received a high score of 87 and a judge score of 9.6, with comments stating it was "Conceptually strong but results need more benchmarking." This indicates that judges recognized the strength of the concept, even if the results could be further validated. Overall, the judge comments reflect appreciation for the innovative ideas and the technical maturity of some projects within fintech, while noting areas for improvement such as benchmarking and evaluation metrics.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [44]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Legal / Compliance," which is mentioned twice in the sample. However, the data sample is limited, and there might be other domains with higher frequency overall. \n\nGiven only this information, I cannot conclusively determine the most common project domain overall. \n\nIf you have a larger dataset or additional context, I can help analyze that to find the exact most common domain.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, "BioForge" is a medical imaging solution in the security domain, and "Neural Canvas" is a low-latency inference system for multimodal agents in autonomous systems, also within the security domain.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally had positive feedback about the fintech projects. For example, they described "WealthifyAI 16" as having a "comprehensive and technically mature approach," and "AutoMate 5" was noted for being "forward-looking with solid supporting data." Additionally, "TrendLens 19" was described as "technically ambitious and well-executed." Overall, the judges appreciated the technical quality, ambition, and potential impact of the fintech projects.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?



##### ✅ Answer

With short, repetitive sentences (like FAQs), semantic chunking will likely over-chunk or create poorly differentiated chunks. Here's why:

Low Semantic Variance: All sentences have similar embedding vectors because they share much of the same vocabulary and structure.

"What is your return policy?"

"What is your shipping policy?"

"What is your warranty policy?"

These will have nearly identical embeddings despite being different questions.

 Semantic chunking relies on detecting "semantic shifts" between sentences, but with repetitive content, there are no clear boundaries.

The algorithm might group questions by superficial patterns rather than actual meaning.

Pure semantic chunking fails with highly repetitive content because it lacks meaningful variance.

Domain knowledge is crucial for FAQs, use the fact that they're question-answer pairs with predictable patterns.

Smaller, intent-based chunks work better than trying to find "natural" semantic boundaries that don't exist.


    for doc in documents:
        content = doc.page_content
        # Split into individual FAQ items
        faq_items = content.split('\n\n')  # Adjust based on your data structure
        
        # Group by intent
        intent_chunks = intent_based_chunking(faq_items)
        
        # Create new documents for each intent group
        for chunk in intent_chunks:
            if chunk:  # Skip empty chunks
                processed_docs.append(Document(
                    page_content='\n'.join(chunk),
                    metadata=doc.metadata  # Preserve original metadata
                ))


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [50]:
### YOUR CODE HERE
from ragas.testset import TestsetGenerator
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import pandas as pd

# Load and prepare your base documents
loader = TextLoader("your_knowledge_base.txt")
documents = loader.load()

# Create knowledge graph relationships (if applicable)
knowledge_graph = {
    "entities": ["product_features", "pricing_tiers", "user_roles"],
    "relationships": ["depends_on", "alternative_to", "requires"]
}

# Generate synthetic test set
generator = TestsetGenerator.from_default(
    generator_llm=ChatOpenAI(model="gpt-3.5-turbo"),
    critic_llm=ChatOpenAI(model="gpt-4"),
    embeddings=embeddings
)

# Define question distribution
testset_distribution = {
    "simple": 0.3,      # Factual lookup questions
    "reasoning": 0.4,   # Multi-hop reasoning
    "multi_context": 0.2, # Requires multiple documents
    "conditional": 0.1   # If-then scenarios
}

# Generate test set
testset = generator.generate(
    documents, 
    test_size=50,  # Adjust based on needs
    distribution=testset_distribution,
    with_debugging_logs=True
)

# Convert to evaluation format
golden_dataset = []
for example in testset.examples:
    golden_dataset.append({
        "question": example.question,
        "ground_truth": example.answer,
        "reference_contexts": example.reference_contexts,
        "question_type": example.question_type
    })
golden_dataset

ModuleNotFoundError: No module named 'ragas'